# 3. Model Evaluation & Testing

Now let's actually check how well the fine-tuned model does, using the standard SQuAD
metrics — **Exact Match (EM)** and **F1 score** — and look at some real predictions
side by side with the ground truth answers.

If you skipped notebook 2 (fine-tuning), this notebook automatically falls back to a
pretrained SQuAD checkpoint from the HuggingFace Hub so you can still see how
evaluation works end-to-end.

In [7]:
from pathlib import Path
from datasets import load_dataset
from transformers import pipeline, AutoTokenizer, AutoModelForQuestionAnswering
import torch

MODEL_DIR = Path("../models/bert-squad-finetuned")
FALLBACK_MODEL = "distilbert-base-cased-distilled-squad"
NUM_EVAL_EXAMPLES = 300

if MODEL_DIR.exists() and any(MODEL_DIR.iterdir()):
    model_source = str(MODEL_DIR)
    print(f"Evaluating locally fine-tuned model at: {model_source}")
else:
    model_source = FALLBACK_MODEL
    print(f"No local fine-tuned model found — evaluating fallback pretrained model: {model_source}")

# Reuse existing variables if they are already defined in the notebook.
if "tokenizer" not in globals():
    tokenizer = AutoTokenizer.from_pretrained(model_source)
if "model" not in globals():
    model = AutoModelForQuestionAnswering.from_pretrained(model_source)

try:
    qa_pipeline = pipeline("question-answering", model=model, tokenizer=tokenizer)
except KeyError:

    def qa_pipeline(question: str, context: str):
        inputs = tokenizer(question, context, return_tensors="pt", truncation=True)
        with torch.no_grad():
            outputs = model(**inputs)

        start_logits = outputs.start_logits[0]
        end_logits = outputs.end_logits[0]

        start_idx = int(torch.argmax(start_logits))
        end_idx = int(torch.argmax(end_logits)) + 1

        # Keep span valid
        if end_idx <= start_idx:
            end_idx = start_idx + 1

        answer_ids = inputs["input_ids"][0][start_idx:end_idx]
        answer = tokenizer.decode(answer_ids, skip_special_tokens=True).strip()

        # Rough confidence score using softmax over start/end logits
        start_probs = torch.softmax(start_logits, dim=-1)
        end_probs = torch.softmax(end_logits, dim=-1)
        score = float(start_probs[start_idx].item() * end_probs[end_idx - 1].item())

        return {"answer": answer, "score": score}


Evaluating locally fine-tuned model at: ..\models\bert-squad-finetuned


## Load a validation subset to evaluate on

In [9]:
# Use the canonical namespaced dataset repo id to avoid the HF URI parsing issue
# caused by some environments when loading the bare "squad" alias.
raw_datasets = load_dataset("rajpurkar/squad")
eval_examples = raw_datasets["validation"].shuffle(seed=42).select(range(NUM_EVAL_EXAMPLES))
eval_examples[0]

{'id': '572759665951b619008f8884',
 'title': 'Private_school',
 'context': 'Private schooling in the United States has been debated by educators, lawmakers and parents, since the beginnings of compulsory education in Massachusetts in 1852. The Supreme Court precedent appears to favor educational choice, so long as states may set standards for educational accomplishment. Some of the most relevant Supreme Court case law on this is as follows: Runyon v. McCrary, 427 U.S. 160 (1976); Wisconsin v. Yoder, 406 U.S. 205 (1972); Pierce v. Society of Sisters, 268 U.S. 510 (1925); Meyer v. Nebraska, 262 U.S. 390 (1923).',
 'question': 'In what year did Massachusetts first require children to be educated in schools?',
 'answers': {'text': ['1852', '1852', '1852'],
  'answer_start': [158, 158, 158]}}

## SQuAD metrics: Exact Match & F1

These are simple, well-known implementations of the official SQuAD evaluation script —
just plain Python (`re`, `string`, `collections`), no extra dependencies needed.

In [10]:
def normalize_answer(s: str) -> str:
    """Lowercase, remove punctuation, articles, and extra whitespace — standard
    SQuAD-style normalization so 'The Eiffel Tower.' and 'eiffel tower' are treated
    as the same answer."""
    
    def remove_articles(text):
        return re.sub(r"\b(a|an|the)\b", " ", text)

    def white_space_fix(text):
        return " ".join(text.split())

    def remove_punc(text):
        exclude = set(string.punctuation)
        return "".join(ch for ch in text if ch not in exclude)

    def lower(text):
        return text.lower()

    return white_space_fix(remove_articles(remove_punc(lower(s))))


def exact_match_score(prediction: str, ground_truth: str) -> int:
    return int(normalize_answer(prediction) == normalize_answer(ground_truth))


def f1_score(prediction: str, ground_truth: str) -> float:
    pred_tokens = normalize_answer(prediction).split()
    truth_tokens = normalize_answer(ground_truth).split()

    common = Counter(pred_tokens) & Counter(truth_tokens)
    num_same = sum(common.values())

    if len(pred_tokens) == 0 or len(truth_tokens) == 0:
        # Edge case: if either is empty, they're only equal if both are empty
        return int(pred_tokens == truth_tokens)
    if num_same == 0:
        return 0.0

    precision = num_same / len(pred_tokens)
    recall = num_same / len(truth_tokens)
    return (2 * precision * recall) / (precision + recall)


def metric_max_over_ground_truths(metric_fn, prediction, ground_truths):
    return max(metric_fn(prediction, gt) for gt in ground_truths)

## Run predictions and compute metrics over the eval subset

In [11]:
em_total = 0.0
f1_total = 0.0
results_for_display = []

for example in eval_examples:
    prediction = qa_pipeline(question=example["question"], context=example["context"])
    predicted_answer = prediction["answer"]
    ground_truths = example["answers"]["text"]

    em_total += metric_max_over_ground_truths(exact_match_score, predicted_answer, ground_truths)
    f1_total += metric_max_over_ground_truths(f1_score, predicted_answer, ground_truths)

    results_for_display.append(
        {
            "question": example["question"],
            "predicted_answer": predicted_answer,
            "ground_truth": ground_truths[0],
            "confidence": round(prediction["score"], 3),
        }
    )

n = len(eval_examples)
em = 100.0 * em_total / n
f1 = 100.0 * f1_total / n

print(f"Evaluated on {n} examples")
print(f"Exact Match: {em:.2f}")
print(f"F1 Score:    {f1:.2f}")

Evaluated on 300 examples
Exact Match: 22.33
F1 Score:    33.82


## Sample predictions

A quick look at individual predictions vs the ground truth — useful for a portfolio
write-up or interview to show the model isn't a black box.

In [12]:
import pandas as pd

results_df = pd.DataFrame(results_for_display)
results_df.head(15)

,question,predicted_answer,ground_truth,confidence
0,In what year did Massachusetts first require c...,1852,1852,0.150
1,When were stromules discovered?,oddly,1962,0.004
2,Which artist who had a major influence on the ...,horace walpole,Horace Walpole,0.194
3,"In 1890, who did the university decide to team...",shi,several regional colleges and universities,0.004
4,Who got a touchdown making the score 10-7?,jonathan stewart,Jonathan Stewart,0.147
5,How many Examination Boards exist in India?,30,30,0.222
6,Who started rumors in 2008 that ABC would sell...,"august 15, 2008, disney",Caris & Co.,0.069
7,Which network broadcasted the 50th Super Bowl ...,cbs broadcast super bowl 50,CBS,0.112
8,Why was this short termed organization created?,william e. simon,coordinate the response to the embargo,0.193
9,What does LGM stands for?,reduced,Last Glacial Maximum,0.000


## Interpreting the results

- If you're using the **fine-tuned model from notebook 2** (trained on a small subset,
  1 epoch, CPU): expect EM in roughly the 55-65 range and F1 in the 68-75 range. That's
  a solid result for a lightweight CPU-only training run, well below the official SQuAD
  leaderboard (full BERT-large: EM ~87 / F1 ~93) because of the reduced data and epochs.
- If you're using the **fallback pretrained model** (already fine-tuned on the *full*
  SQuAD dataset by the community): expect noticeably higher scores, since it saw the
  entire training set with proper compute.
- Most disagreements between predicted and ground-truth answers come from **partial
  span overlap** — e.g. predicting "Gustave Eiffel" vs ground truth "Eiffel" — which F1
  captures more gracefully than the stricter Exact Match metric.

This gap is exactly why the web app defaults to the fully-trained fallback model for
demo purposes, while these notebooks show the real fine-tuning process end-to-end.

## Try your own question

A quick manual sanity check outside the formal eval loop.

In [13]:
context = (
    "The Great Barrier Reef is the world's largest coral reef system, composed of "
    "over 2,900 individual reefs and 900 islands stretching for over 2,300 kilometers "
    "over an area of approximately 344,400 square kilometers. It is located in the "
    "Coral Sea, off the coast of Queensland, Australia."
)
question = "Where is the Great Barrier Reef located?"

result = qa_pipeline(question=question, context=context)
print(f"Q: {question}")
print(f"A: {result['answer']}  (confidence: {result['score']:.3f})")

Q: Where is the Great Barrier Reef located?
A: 2, 900 individual reefs and 900 islands stretching for over 2, 300 kilometers over an area of approximately 344, 400 square kilometers. it is located in the coral sea  (confidence: 0.034)


## Next step

The model is ready to demo — run `python app.py` from the project root and try it out
through the web UI instead of the notebook.